# UACB-604 · Lecture 2.3 — Long reads, consensus and barcodes

**You will:**
1. Stream real **Oxford Nanopore** reads (MinION, *E. coli* clinical isolate, run DRR242229) and compare them with Illumina reads
2. Compute **N50**, the standard summary of read (and assembly) length
3. Simulate how PacBio **HiFi** turns noisy passes into accurate reads by majority vote
4. Use the **birthday problem** to see why linked-read barcodes sometimes collide

In [ ]:
import gzip, urllib.request, math
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(604)

def stream_fastq(url, n_reads):
    req = urllib.request.Request(url, headers={"User-Agent": "uacb604-course"})
    with gzip.GzipFile(fileobj=urllib.request.urlopen(req, timeout=60)) as fh:
        lines = (l.decode().rstrip("\n") for l in fh)
        for _ in range(n_reads):
            h, s, plus, q = (next(lines) for _ in range(4))
            yield h, s, q

ont = list(stream_fastq("https://ftp.sra.ebi.ac.uk/vol1/fastq/DRR242/DRR242229/DRR242229_1.fastq.gz", 3000))
lengths = np.array([len(s) for _, s, _ in ont])
meanQ = np.array([np.mean([ord(c) - 33 for c in q]) for _, _, q in ont])
print(f"{len(ont)} reads | shortest {lengths.min():,} | median {np.median(lengths):,.0f} | longest {lengths.max():,} bp")

## 1. Read-length distribution
Long-read lengths span orders of magnitude, so we plot them on a **log scale** (Lecture 1.1).

In [ ]:
plt.hist(lengths, bins=np.logspace(2, 5, 60), color="slateblue")
plt.xscale("log"); plt.xlabel("read length (bp, log scale)"); plt.ylabel("reads")
plt.axvline(63, color="red", ls="--", label="Illumina read in 2.2 (63 bp)"); plt.legend(); plt.show()

## 2. N50
Sort reads from longest to shortest and add up their lengths. **N50** is the length of the read at which you pass half of all bases.
It answers: *"half of my data is in reads at least this long."* The same idea measures assembly quality in Unit 7.

In [ ]:
def n50(values):
    v = np.sort(values)[::-1]
    cum = np.cumsum(v)
    return int(v[np.searchsorted(cum, cum[-1] / 2)])

print("mean length:", int(lengths.mean()), "| median:", int(np.median(lengths)), "| N50:", n50(lengths))

✏️ **Exercise 1.** Why is N50 larger than both the mean and the median here? Compute N50 for the made-up read set `[2, 2, 2, 2, 10]` kb by hand, then with the function.

In [ ]:
# your code here

## 3. Quality of these Nanopore reads

In [ ]:
plt.hist(meanQ, bins=40, color="darkorange")
plt.xlabel("mean Phred quality per read"); plt.ylabel("reads"); plt.show()
print(f"median of per-read mean Q: {np.median(meanQ):.1f}")

⚠️ Averaging Q values is a quick summary but not the true error rate. Errors live on the *probability* scale: average $10^{-Q/10}$, then convert back.

In [ ]:
q = np.array([ord(c) - 33 for c in ont[0][2]])
print("mean of Q:", q.mean().round(1), "  Q of the mean error:", (-10 * np.log10(np.mean(10 ** (-q / 10)))).round(1))

✏️ **Exercise 2.** Why is the "Q of the mean error" lower than the "mean of Q"? (Hint: a few very bad bases dominate the average error probability. Compare mean vs median in the primer.)

## 4. HiFi: accuracy from repetition
PacBio HiFi reads come from a circular molecule read **many times** (passes). Suppose each pass gets a given base wrong with probability $e$, independently.
With a **majority vote** over $n$ passes (n odd), the consensus is wrong only if more than half the passes are wrong:
$$P(\text{consensus wrong}) = \sum_{k > n/2} \binom{n}{k} e^k (1-e)^{n-k}$$

In [ ]:
def majority_error(e, n):
    return sum(math.comb(n, k) * e**k * (1 - e)**(n - k) for k in range(n // 2 + 1, n + 1))

for n in (1, 3, 5, 9, 15):
    p = majority_error(0.10, n)
    print(f"{n:2d} passes: consensus error = {p:.2e}  →  Q{-10*math.log10(p):.0f}")

This is a simplification: real errors are mostly insertions/deletions and not fully independent, so PacBio uses a statistical model, not a plain vote. But the *trend* (more passes, more accuracy) is exactly why HiFi reads reach Q30+.

✏️ **Exercise 3.** Simulate the vote with random numbers for e = 0.1 and n = 5 (100,000 trials) and compare with the formula.

In [ ]:
# your code here

## 5. Linked reads and the birthday problem
In linked-read methods (10x Genomics, TELL-Seq) each long DNA molecule gets a **barcode**, and all short reads from it carry that barcode.
If $m$ molecules are spread over $N$ possible barcodes, how often do two molecules get the **same** barcode?

This is the **birthday problem**: in a class of 23 people, the chance that two share a birthday is already over 50%.
$$P(\text{no shared barcode}) \approx e^{-m(m-1)/(2N)}$$

In [ ]:
def p_collision(m, N):
    return 1 - math.exp(-m * (m - 1) / (2 * N))

print(f"birthdays, 23 people: {p_collision(23, 365):.2f}")
N = 1_000_000
for m in (100, 1000, 5000):
    sims = np.mean([len(set(rng.integers(0, N, m))) < m for _ in range(2000)])
    print(f"{m:5d} molecules, {N:,} barcodes: formula {p_collision(m, N):.3f}, simulation {sims:.3f}")

✏️ **Exercise 4.** Collisions are not fatal: two molecules from far-apart parts of the genome sharing a barcode can be detected. Expected number of colliding *pairs* is $\binom{m}{2}/N$. With 1 million barcodes and 1 million molecules, how many pairs share a barcode? Why do linked-read protocols load only a small amount of DNA per barcode?

---
**Further reading:** Pevsner, ch. 9 · Brown, *Genomes 5*, ch. 4.